In [64]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

In [65]:
# Number of users in our fictional app
n_users = 50000

# Create unique user IDs
user_ids = [f"U{i:05d}" for i in range(1, n_users + 1)]

# Generate random signup dates
start_date = datetime(2026, 1, 1)
end_date = datetime(2026, 6, 30)

days_range = (end_date - start_date).days

signup_dates = [
    start_date + timedelta(days=int(x))
    for x in np.random.randint(0, days_range + 1, n_users)
]

print("Number of users:", len(user_ids))
print("First 5 IDs:", user_ids[:5])
print("First 5 signup dates:", signup_dates[:5])

Number of users: 50000
First 5 IDs: ['U00001', 'U00002', 'U00003', 'U00004', 'U00005']
First 5 signup dates: [datetime.datetime(2026, 4, 13, 0, 0), datetime.datetime(2026, 6, 29, 0, 0), datetime.datetime(2026, 4, 3, 0, 0), datetime.datetime(2026, 1, 15, 0, 0), datetime.datetime(2026, 4, 17, 0, 0)]


In [66]:
# Generate user characteristics

countries = np.random.choice(
    ["Colombia", "Mexico", "Argentina", "Chile", "Peru"],
    size=n_users,
    p=[0.40, 0.20, 0.15, 0.15, 0.10]
)

devices = np.random.choice(
    ["Android", "iOS", "Web"],
    size=n_users,
    p=[0.55, 0.30, 0.15]
)

acquisition_channels = np.random.choice(
    ["Instagram", "TikTok", "Google Ads", "Organic", "Referral"],
    size=n_users,
    p=[0.25, 0.25, 0.20, 0.20, 0.10]
)

Generación del conjunto de datos de usuarios


En esta sección se genera el conjunto de datos de usuarios de StyleMatch.

Cada fila representa un usuario registrado e incluye información sobre su
fecha de registro, país, dispositivo y canal de adquisición.

Se generan 50.000 usuarios sintéticos para simular la base de usuarios
de la aplicación.

In [67]:
users = pd.DataFrame({
    "user_id": user_ids,
    "signup_date": signup_dates,
    "country": countries,
    "device": devices,
    "acquisition_channel": acquisition_channels
})

users.head()

,user_id,signup_date,country,device,acquisition_channel
0,U00001,2026-04-13,Colombia,Web,Organic
1,U00002,2026-06-29,Colombia,Web,Google Ads
2,U00003,2026-04-03,Mexico,iOS,Google Ads
3,U00004,2026-01-15,Peru,iOS,Instagram
4,U00005,2026-04-17,Peru,Android,Organic


Generación de eventos de comportamiento

En esta sección se simulan las interacciones de los usuarios con StyleMatch.

Los eventos representan las principales acciones realizadas dentro del producto,
desde la visita inicial hasta la generación y guardado de outfits.

Estos datos permitirán analizar el funnel de conversión, identificar puntos de
abandono y estudiar el comportamiento de los usuarios.

In [68]:
# Lista donde almacenaremos todos los eventos generados
events = []

# Probabilidades de avanzar en cada etapa del funnel
p_signup = 0.80
p_upload = 0.70
p_generate = 0.75
p_save = 0.60
p_return = 0.45

print("Estructura de eventos creada.")

Estructura de eventos creada.


In [69]:
# Generar el recorrido de cada usuario por el funnel

for _, user in users.iterrows():

    user_id = user["user_id"]
    signup_date = user["signup_date"]

    # Cada usuario comienza con una visita
    event_time = signup_date - timedelta(
        minutes=np.random.randint(1, 60)
    )

    events.append({
        "user_id": user_id,
        "event": "visit",
        "timestamp": event_time
    })

    # Registro
    if np.random.random() < p_signup:

        event_time += timedelta(minutes=np.random.randint(1, 15))

        events.append({
            "user_id": user_id,
            "event": "signup",
            "timestamp": event_time
        })

        # Subir una prenda
        if np.random.random() < p_upload:

            event_time += timedelta(minutes=np.random.randint(1, 20))

            events.append({
                "user_id": user_id,
                "event": "upload_clothing",
                "timestamp": event_time
            })

            # Generar outfit
            if np.random.random() < p_generate:

                event_time += timedelta(minutes=np.random.randint(1, 10))

                events.append({
                    "user_id": user_id,
                    "event": "generate_outfit",
                    "timestamp": event_time
                })

                # Guardar outfit
                if np.random.random() < p_save:

                    event_time += timedelta(minutes=np.random.randint(1, 5))

                    events.append({
                        "user_id": user_id,
                        "event": "save_outfit",
                        "timestamp": event_time
                    })

print("Número total de eventos:", len(events))

Número total de eventos: 151622


In [70]:
# Convertir la lista de eventos en un DataFrame

events_df = pd.DataFrame(events)

# Ordenar los eventos cronológicamente
events_df = events_df.sort_values(
    by=["user_id", "timestamp"]
).reset_index(drop=True)

print("Número de eventos:", len(events_df))
print("Número de usuarios:", events_df["user_id"].nunique())

events_df.head(10)

Número de eventos: 151622
Número de usuarios: 50000


,user_id,event,timestamp
0,U00001,visit,2026-04-12 23:03:00
1,U00002,visit,2026-06-28 23:17:00
2,U00002,signup,2026-06-28 23:21:00
3,U00002,upload_clothing,2026-06-28 23:37:00
4,U00002,generate_outfit,2026-06-28 23:39:00
5,U00002,save_outfit,2026-06-28 23:41:00
6,U00003,visit,2026-04-02 23:15:00
7,U00003,signup,2026-04-02 23:23:00
8,U00004,visit,2026-01-14 23:46:00
9,U00004,signup,2026-01-14 23:52:00


In [71]:
# Contar usuarios únicos en cada etapa del funnel

funnel_order = [
    "visit",
    "signup",
    "upload_clothing",
    "generate_outfit",
    "save_outfit"
]

funnel_counts = (
    events_df.groupby("event")["user_id"]
    .nunique()
    .reindex(funnel_order)
)

funnel_counts

event
visit              50000
signup             39966
upload_clothing    27988
generate_outfit    21008
save_outfit        12660
Name: user_id, dtype: int64

In [72]:
# Calcular conversión y abandono entre etapas del funnel

funnel_analysis = pd.DataFrame({
    "users": funnel_counts
})

funnel_analysis["conversion_rate"] = (
    funnel_analysis["users"]
    / funnel_analysis["users"].shift(1)
    * 100
)

funnel_analysis["dropoff_rate"] = (
    100 - funnel_analysis["conversion_rate"]
)

# La primera etapa no tiene una etapa anterior
funnel_analysis.loc["visit", "conversion_rate"] = 100
funnel_analysis.loc["visit", "dropoff_rate"] = 0

funnel_analysis.round(2)

,users,conversion_rate,dropoff_rate
event,,,
visit,50000,100.00,0.00
signup,39966,79.93,20.07
upload_clothing,27988,70.03,29.97
generate_outfit,21008,75.06,24.94
save_outfit,12660,60.26,39.74


Análisis inicial del funnel

El funnel muestra una disminución progresiva de usuarios a medida que avanzan por las etapas principales de StyleMatch.

De los 50.000 usuarios que visitaron la aplicación, 39.966 completaron el registro, lo que representa una conversión del 79,93 %. Posteriormente, 27.988 usuarios subieron una prenda y 21.008 llegaron a generar un outfit.

El mayor punto de abandono se encuentra entre la generación y el guardado del outfit: solo el 60,26 % de los usuarios que generan un outfit decide guardarlo, lo que representa un abandono del 39,74 %.

Este comportamiento identifica la etapa `generate_outfit → save_outfit` como una oportunidad de análisis y optimización. A partir de este hallazgo podrían evaluarse hipótesis relacionadas con la calidad de las recomendaciones, la experiencia de guardado o el valor percibido por el usuario.

## Análisis de retención de usuarios

Además de analizar la conversión inicial, es importante conocer si los usuarios regresan a StyleMatch después de su primera experiencia.

En esta sección se simulará el retorno de los usuarios en diferentes momentos después de su registro, permitiendo analizar métricas de retención como Day 1, Day 7 y Day 30.

Este análisis permitirá evaluar la capacidad del producto para mantener activos a sus usuarios a lo largo del tiempo.

In [73]:
# Usuarios que completaron el registro

registered_users = events_df[
    events_df["event"] == "signup"
]["user_id"].unique()

print("Usuarios registrados:", len(registered_users))

Usuarios registrados: 39966


In [74]:
# Probabilidades de retención
retention_probabilities = {
    "day_1": 0.55,
    "day_7": 0.35,
    "day_30": 0.20
}

retention_data = []

# Simular si cada usuario regresa en Day 1, Day 7 y Day 30
for user_id in registered_users:

    retained_day_1 = np.random.random() < retention_probabilities["day_1"]
    retained_day_7 = np.random.random() < retention_probabilities["day_7"]
    retained_day_30 = np.random.random() < retention_probabilities["day_30"]

    retention_data.append({
        "user_id": user_id,
        "day_1": retained_day_1,
        "day_7": retained_day_7,
        "day_30": retained_day_30
    })

print("Registros de retención generados:", len(retention_data))

Registros de retención generados: 39966


In [75]:
# Convertir los datos de retención en un DataFrame
retention_df = pd.DataFrame(retention_data)

# Visualizar los primeros registros
retention_df.head(10)

,user_id,day_1,day_7,day_30
0,U00002,False,True,False
1,U00003,True,False,False
2,U00004,True,False,False
3,U00005,False,False,False
4,U00006,False,False,False
5,U00008,True,False,True
6,U00009,False,True,False
7,U00010,True,False,False
8,U00011,True,False,False
9,U00015,True,True,False


In [76]:
# Calcular las tasas de retención observadas
retention_rates = (
    retention_df[["day_1", "day_7", "day_30"]]
    .mean()
    .mul(100)
    .round(2)
)

retention_rates

day_1     54.81
day_7     34.72
day_30    19.57
dtype: float64

### Resultados de retención

A partir de los usuarios que completaron el registro, se analizaron las tasas de
retención en distintos momentos del ciclo de vida del usuario.

Los resultados obtenidos fueron:

- **Day 1:** 54.81%
- **Day 7:** 34.72%
- **Day 30:** 19.57%

Se observa una disminución progresiva de la retención a medida que pasa el tiempo.
Mientras más de la mitad de los usuarios regresan durante el primer día, solamente
cerca del 20% continúa regresando después de 30 días.

Este comportamiento sugiere que una oportunidad importante para el producto sería
trabajar en estrategias que incentiven a los usuarios a regresar después de su
primera experiencia.

In [77]:
# Unir los eventos con la información de cada usuario
events_enriched = events_df.merge(
    users[["user_id", "country", "device", "acquisition_channel"]],
    on="user_id",
    how="left"
)

events_enriched.head(10)

,user_id,event,timestamp,country,device,acquisition_channel
0,U00001,visit,2026-04-12 23:03:00,Colombia,Web,Organic
1,U00002,visit,2026-06-28 23:17:00,Colombia,Web,Google Ads
2,U00002,signup,2026-06-28 23:21:00,Colombia,Web,Google Ads
3,U00002,upload_clothing,2026-06-28 23:37:00,Colombia,Web,Google Ads
4,U00002,generate_outfit,2026-06-28 23:39:00,Colombia,Web,Google Ads
5,U00002,save_outfit,2026-06-28 23:41:00,Colombia,Web,Google Ads
6,U00003,visit,2026-04-02 23:15:00,Mexico,iOS,Google Ads
7,U00003,signup,2026-04-02 23:23:00,Mexico,iOS,Google Ads
8,U00004,visit,2026-01-14 23:46:00,Peru,iOS,Instagram
9,U00004,signup,2026-01-14 23:52:00,Peru,iOS,Instagram


In [78]:
# Contar usuarios únicos por dispositivo y etapa del funnel
funnel_by_device = (
    events_enriched
    .groupby(["device", "event"])["user_id"]
    .nunique()
    .unstack()
)

# Ordenar las etapas del funnel
funnel_by_device = funnel_by_device[funnel_order]

funnel_by_device

event,visit,signup,upload_clothing,generate_outfit,save_outfit
device,,,,,
Android,27611,22034,15505,11623,6954
Web,7494,5956,4209,3163,1936
iOS,14895,11976,8274,6222,3770


In [79]:
# Calcular la conversión total desde visita hasta cada etapa
conversion_by_device = (
    funnel_by_device
    .div(funnel_by_device["visit"], axis=0)
    .mul(100)
    .round(2)
)

conversion_by_device

event,visit,signup,upload_clothing,generate_outfit,save_outfit
device,,,,,
Android,100.0,79.80,56.16,42.10,25.19
Web,100.0,79.48,56.16,42.21,25.83
iOS,100.0,80.40,55.55,41.77,25.31


In [80]:
p_signup = 0.80
p_upload = 0.70
p_generate = 0.75
p_save = 0.60

### Conversión por dispositivo

Al comparar el funnel entre Android, iOS y Web no se observan diferencias
significativas en la conversión.

La conversión desde visita hasta guardado del outfit se mantiene alrededor del
25% en los tres dispositivos.

Esto indica que, dentro de los datos simulados, el tipo de dispositivo no parece
ser un factor relevante para explicar el abandono observado en el funnel.

Por lo tanto, el análisis continuará explorando otras dimensiones, como el canal
de adquisición y el país del usuario.

## Exportación de datasets

Una vez generados y validados los datos sintéticos, se exportan los principales
datasets del proyecto para utilizarlos posteriormente en el análisis de producto.

Los archivos se almacenan en `data/raw`, conservando los datos originales
generados antes de realizar transformaciones o análisis adicionales.

In [81]:
users.to_csv("../data/raw/users.csv", index=False)

print("users.csv guardado correctamente.")

users.csv guardado correctamente.


In [82]:
events_df.to_csv("../data/raw/events.csv", index=False)

print("events.csv guardado correctamente.")

events.csv guardado correctamente.


In [83]:
retention_df.to_csv("../data/raw/retention.csv", index=False)

print("retention.csv guardado correctamente.")

retention.csv guardado correctamente.


# Análisis de Producto — StyleMatch

## Objetivo del análisis

En este notebook se analizará el comportamiento de los usuarios de StyleMatch
a partir de los datos generados previamente.

El objetivo es identificar patrones de comportamiento, puntos de abandono
dentro del funnel de conversión y diferencias entre segmentos de usuarios.

El análisis se enfocará principalmente en:

- Conversión a través del funnel.
- Retención de usuarios.
- Comportamiento por canal de adquisición.
- Comportamiento por país y dispositivo.
- Identificación de oportunidades de mejora del producto.

In [84]:
import pandas as pd
import numpy as np

In [85]:
users = pd.read_csv("../data/raw/users.csv")
events = pd.read_csv("../data/raw/events.csv")
retention = pd.read_csv("../data/raw/retention.csv")

print("Users:", users.shape)
print("Events:", events.shape)
print("Retention:", retention.shape)

Users: (50000, 5)
Events: (151622, 3)
Retention: (39966, 4)


## Validación de calidad de los datos

Antes de realizar el análisis de producto, se verifica la calidad de los datasets
para identificar posibles problemas que puedan afectar los resultados.

Se revisarán los tipos de datos, valores nulos, registros duplicados y la
estructura general de cada conjunto de datos.

In [86]:
users.info()

<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype
---  ------               --------------  -----
 0   user_id              50000 non-null  str  
 1   signup_date          50000 non-null  str  
 2   country              50000 non-null  str  
 3   device               50000 non-null  str  
 4   acquisition_channel  50000 non-null  str  
dtypes: str(5)
memory usage: 1.9 MB


In [87]:
users["signup_date"] = pd.to_datetime(users["signup_date"])

print(users["signup_date"].dtype)

datetime64[us]


In [88]:
# Revisar valores nulos
print("Valores nulos:")
print(users.isnull().sum())

# Revisar usuarios duplicados
print("\nUser IDs duplicados:")
print(users["user_id"].duplicated().sum())

# Revisar categorías existentes
print("\nPaíses:")
print(users["country"].value_counts())

print("\nDispositivos:")
print(users["device"].value_counts())

print("\nCanales de adquisición:")
print(users["acquisition_channel"].value_counts())

Valores nulos:
user_id                0
signup_date            0
country                0
device                 0
acquisition_channel    0
dtype: int64

User IDs duplicados:
0

Países:
country
Colombia     19914
Mexico       10028
Chile         7684
Argentina     7473
Peru          4901
Name: count, dtype: int64

Dispositivos:
device
Android    27611
iOS        14895
Web         7494
Name: count, dtype: int64

Canales de adquisición:
acquisition_channel
TikTok        12552
Instagram     12336
Organic       10112
Google Ads     9918
Referral       5082
Name: count, dtype: int64


### Resultados de la validación de usuarios

El dataset de usuarios contiene 50.000 registros únicos y no presenta valores
nulos ni identificadores de usuario duplicados.

Las variables categóricas presentan valores consistentes para país, dispositivo
y canal de adquisición, sin categorías inesperadas.

La variable `signup_date`, inicialmente importada como texto, fue convertida a
formato `datetime` para permitir análisis temporales posteriores.

Por lo tanto, el dataset de usuarios se considera consistente y adecuado para
continuar con el análisis.

In [89]:
events.info()

<class 'pandas.DataFrame'>
RangeIndex: 151622 entries, 0 to 151621
Data columns (total 3 columns):
 #   Column     Non-Null Count   Dtype
---  ------     --------------   -----
 0   user_id    151622 non-null  str  
 1   event      151622 non-null  str  
 2   timestamp  151622 non-null  str  
dtypes: str(3)
memory usage: 3.5 MB


In [90]:
events["timestamp"] = pd.to_datetime(events["timestamp"])

print(events["timestamp"].dtype)

datetime64[us]


In [91]:
# Revisar tipos de eventos
print("Tipos de eventos:")
print(events["event"].value_counts())

# Revisar registros duplicados
print("\nRegistros duplicados:")
print(events.duplicated().sum())

Tipos de eventos:
event
visit              50000
signup             39966
upload_clothing    27988
generate_outfit    21008
save_outfit        12660
Name: count, dtype: int64

Registros duplicados:
0


In [92]:
events.duplicated().sum()

np.int64(0)

In [93]:
retention.info()

<class 'pandas.DataFrame'>
RangeIndex: 39966 entries, 0 to 39965
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   user_id  39966 non-null  str  
 1   day_1    39966 non-null  bool 
 2   day_7    39966 non-null  bool 
 3   day_30   39966 non-null  bool 
dtypes: bool(3), str(1)
memory usage: 429.5 KB


In [94]:
# Revisar valores nulos
print("Valores nulos:")
print(retention.isnull().sum())

# Revisar usuarios duplicados
print("\nUser IDs duplicados:")
print(retention["user_id"].duplicated().sum())

# Revisar distribución de las variables de retención
print("\nRetención Day 1:")
print(retention["day_1"].value_counts())

print("\nRetención Day 7:")
print(retention["day_7"].value_counts())

print("\nRetención Day 30:")
print(retention["day_30"].value_counts())

Valores nulos:
user_id    0
day_1      0
day_7      0
day_30     0
dtype: int64

User IDs duplicados:
0

Retención Day 1:
day_1
True     21904
False    18062
Name: count, dtype: int64

Retención Day 7:
day_7
False    26089
True     13877
Name: count, dtype: int64

Retención Day 30:
day_30
False    32145
True      7821
Name: count, dtype: int64


### Resultados de la validación de retención

El dataset de retención contiene 39.966 usuarios registrados y no presenta
valores nulos ni identificadores de usuario duplicados.

Las variables de retención presentan una disminución progresiva a medida que
transcurre el tiempo:

- Day 1: aproximadamente 54,8 % de los usuarios regresan.
- Day 7: aproximadamente 34,7 % de los usuarios regresan.
- Day 30: aproximadamente 19,6 % de los usuarios regresan.

Los resultados son consistentes con el comportamiento esperado de los datos
simulados y el dataset se considera adecuado para continuar con el análisis.